<a href="https://colab.research.google.com/github/klizaola/Predicting-Pollinator-Syndrome/blob/main/PollinatorProject_clean_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading all of our libraries:

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn import model_selection
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

Importing our cleaned .csv file into Colab:

In [4]:
from google.colab import files
uploaded = files.upload("Dryad dataset_final_clean.csv")
dryad_data = pd.read_csv(next(iter(uploaded)))

Saving Dryad dataset_final_clean.csv to Dryad dataset_final_clean.csv/Dryad dataset_final_clean (1).csv


Getting class counts of our dataset (pollinator types, microbial isolates and plant species) to identify potential bias:

In [5]:
dryad_cols = ['Family', 'Floral visitor', 'Plant species', 'Isolate1', 'Isolate2', 'Isolate3', 'Isolate4', 'Isolate5']

for col in dryad_cols:
  print(f"=== Class Counts for '{col}' ===")
  counts = dryad_data[col].value_counts()
  percentages = dryad_data[col].value_counts(normalize=True) * 100

  distribution = pd.DataFrame({
      'Count': counts,
      'Percentage (%)': percentages.round(2)
  })

  display(distribution.head(10))
  print("\n")

=== Class Counts for 'Family' ===


,Count,Percentage (%)
Family,,
Proteaceae,102,56.04
Iridaceae,19,10.44
Ericaceae,14,7.69
Orobanchaceae,9,4.95
Acanthaceae,7,3.85
Xanthorreaceae,6,3.30
Lamiaceae,6,3.30
Scrophulariaceae,6,3.30
Caryophyllaceae,3,1.65




=== Class Counts for 'Floral visitor' ===


,Count,Percentage (%)
Floral visitor,,
Beetle,102,56.04
Long-tongued insects,38,20.88
Birds,25,13.74
Short-tongued insects,17,9.34




=== Class Counts for 'Plant species' ===


,Count,Percentage (%)
Plant species,,
Protea welwitschii,21,11.54
Protea caffra,14,7.69
Protea roupelliae,13,7.14
Protea dracomontana,13,7.14
Protea subvestita,12,6.59
Protea repens,10,5.49
Protea laurifolia,10,5.49
Glumicalyx goseloides,6,3.30
Protea neriifolia,6,3.30


... (6 outputs left out)



=== Class Counts for 'Isolate4' ===


,Count,Percentage (%)
Isolate4,,
0,166,91.21
Klebsiella_Enterobacter,3,1.65
Hanseniaspora thailandica,2,1.10
Metschnikowia proteae,1,0.55
Rhodotorula taiwanensis,1,0.55
Rosenbergiella,1,0.55
Wickerhamomyces,1,0.55
Paraburkholderia,1,0.55
Rhodotorula mucilaginosa,1,0.55




=== Class Counts for 'Isolate5' ===


,Count,Percentage (%)
Isolate5,,
0,178,97.80
Pseudomonas_2,1,0.55
Pichia,1,0.55
Asaia,1,0.55
Rhodotorula mucilaginosa,1,0.55


Transforming our dataset into a binary matrix:

In [9]:
df = dryad_data.copy()
df.columns = df.columns.str.strip()

isolate_columns = [
    "Isolate1",
    "Isolate2",
    "Isolate3",
    "Isolate4",
    "Isolate5",
]
df["Isolates"] = df[isolate_columns].apply(
    lambda row: [
        str(isolate).strip() for isolate in row
        if pd.notna(isolate) and isolate !=0 and str(isolate).strip() != "0" and str(isolate).strip() != ""
    ],
    axis = 1
)

mlb = MultiLabelBinarizer()
encoded_isolates = pd.DataFrame(
    mlb.fit_transform(df["Isolates"]),
    columns=mlb.classes_,
    index=df.index

)

X = encoded_isolates

y = df["Floral visitor"]

print(X)
print(y)


     Achromobacter  Acinetobacter  Arthrobacter  Asaia  Auriculibuller fuscus  \
0                0              0             0      0                      0   
1                0              0             0      0                      0   
2                0              0             0      0                      0   
3                0              1             0      0                      0   
4                0              0             0      1                      0   
..             ...            ...           ...    ...                    ...   
177              0              0             0      0                      0   
178              0              0             0      0                      0   
179              0              0             0      0                      0   
180              0              0             0      0                      0   
181              0              0             0      0                      0   

     Bacillus_1  Bacillus_2

Splitting the transformed data matrix into training, test, and validation subsets while maintaining groups by plant species to prevent data leakage that would result in the model predicting by plant species instead of by microbe community:

In [10]:
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=df["Plant species"]))

X_train_val, y_train_val = X.iloc[train_val_idx], y.iloc[train_val_idx]
groups_train_val = df["Plant species"].iloc[train_val_idx]

X_test_gp, y_test_gp = X.iloc[test_idx], y.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss_val.split(X_train_val, y_train_val, groups=groups_train_val))

X_train_gp, y_train_gp = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
X_val_gp, y_val_gp = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

train_plants = set(df['Plant species'].iloc[train_val_idx].iloc[train_idx])
val_plants = set(df['Plant species'].iloc[train_val_idx].iloc[val_idx])
test_plants = set(df['Plant species'].iloc[test_idx])

print(f"Total dataset size: {len(X)} samples")
print(f"Training set:   {len(X_train_gp)} samples ({len(X_train_gp)/len(X)*100:.1f}%) | Unique Plant Species: {len(train_plants)}")
print(f"Validation set: {len(X_val_gp)} samples ({len(X_val_gp)/len(X)*100:.1f}%) | Unique Plant Species: {len(val_plants)}")
print(f"Test set:       {len(X_test_gp)} samples ({len(X_test_gp)/len(X)*100:.1f}%) | Unique Plant Species: {len(test_plants)}")

assert len(train_plants.intersection(val_plants)) == 0, "Leakage between Train & Val!"
assert len(train_plants.intersection(test_plants)) == 0, "Leakage between Train & Test!"
print("\nSuccess! No plant species overlap across splits.")

Total dataset size: 182 samples
Training set:   100 samples (54.9%) | Unique Plant Species: 25
Validation set: 44 samples (24.2%) | Unique Plant Species: 9
Test set:       38 samples (20.9%) | Unique Plant Species: 9

Success! No plant species overlap across splits.


Running RandomForestClassifier and reporting balanced accuracy and macro F1 against a simple baseline and against a majority class baseline:

In [14]:
rf_grouped = RandomForestClassifier(
    n_estimators=100,
    random_state=36,
    max_depth=None,
    max_features='sqrt',
    max_leaf_nodes=10,
    class_weight='balanced'
)

rf_grouped.fit(X_train_gp, y_train_gp)

val_preds = rf_grouped.predict(X_val_gp)
test_preds = rf_grouped.predict(X_test_gp)
most_frequent_gp = y_train_gp.mode()[0]
simple_baseline_val = np.full_like(y_val_gp, fill_value=most_frequent_gp, dtype=object)
simple_baseline_test = np.full_like(y_test_gp, fill_value=most_frequent_gp, dtype=object)

print(f"Balanced Accuracy and Macro F1 against a simple baseline:")
print("=== Validation Set Evaluation ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val_gp, val_preds):.4f}")
print(f"Macro F1: {f1_score(y_val_gp, val_preds, average='macro'):.4f}\n")

print("=== Test Set Evaluation ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test_gp, test_preds):.4f}")
print(f"Macro F1: {f1_score(y_test_gp, test_preds, average='macro'):.4f}\n")
print(classification_report(y_test_gp, test_preds, zero_division=0))

print(f"Balanced Accuracy and Macro F1 against a majority class baseline: {most_frequent_gp}\n")
print("=== Validation Evaluation ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val_gp, simple_baseline_val):.4f}")
print(f"Macro F1: {f1_score(y_val_gp, simple_baseline_val, average='macro', zero_division=0):.4f}\n")

print("=== Test Evaluation ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test_gp, simple_baseline_test):.4f}")
print(f"Macro F1: {f1_score(y_test_gp, simple_baseline_test, average='macro', zero_division=0):.4f}")

Balanced Accuracy and Macro F1 against a simple baseline:
=== Validation Set Evaluation ===
Balanced Accuracy: 0.8333
Macro F1: 0.8413

=== Test Set Evaluation ===
Balanced Accuracy: 0.3750
Macro F1: 0.2396

                       precision    recall  f1-score   support

               Beetle       1.00      0.33      0.50        24
                Birds       0.10      0.17      0.12         6
 Long-tongued insects       0.20      1.00      0.33         4
Short-tongued insects       0.00      0.00      0.00         4

             accuracy                           0.34        38
            macro avg       0.33      0.38      0.24        38
         weighted avg       0.67      0.34      0.37        38

Balanced Accuracy and Macro F1 against a majority class baseline: Beetle

=== Validation Evaluation ===
Balanced Accuracy: 0.3333
Macro F1: 0.2648

=== Test Evaluation ===
Balanced Accuracy: 0.2500
Macro F1: 0.1935


Re-grouping our pollinator types in attempt to improve model prediction scores:

Running the RandomForestClassifier again, but this time after combining "Short-tongued insects" with "Long-tongued insects", to create 3 different possible pollinator types: Birds, Beetles, and Tongued-insects